In [1]:
import os
import pdfplumber
import pypdfium2 as pyfium
import chromadb
import time
from docx import Document
from pptx import Presentation
from IPython.display import display,Markdown
from PIL import Image
from dotenv import load_dotenv
from google import genai


In [2]:
load_dotenv()
client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

In [3]:
db_client = chromadb.PersistentClient(
    path= "./Chroma_db"
)

In [4]:
db_client.delete_collection(
    name="multimodal_rag"
)

In [5]:
try:
    collection = db_client.get_collection(
        name = "multimodal_rag"
    )
    print("collection loaded")
except:
    collection = db_client.create_collection(
        name="multimodal_rag"
    )
    print("collection created")

collection created


In [94]:
def extract_docx(file_path):
    doc = Document(file_path)
    text =""
    for paragraph in doc.paragraphs:
        if paragraph.text.strip():
            text += paragraph.text + "\n"
    return text

In [95]:
def get_processed_files(collection):
    results =collection.get()
    processed_files = set()
    for metadata in results["metadatas"]:
        processed_files.add(
            metadata["source"]
        )
    return processed_files

In [96]:
def load_document(file_path):
    extension = os.path.splitext(file_path)[1].lower()
    documents = []
    if extension ==".txt":
        with open(file_path,"r",encoding="utf-8") as file:
            text =file.read()
        if text.strip():
            documents.append(
                {
                    "content":text,
                    "type":"text"
                }
            )
    
    elif extension ==".pdf":
        text =""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
            
                if page_text:
                    text += page_text+ "\n"
        text = text.replace("\u2028", "\n")
        text = text.replace("\u2029", "\n")
        if text.strip():
            documents.append(
                {
                    "content":text,
                    "type":"text"
                }
            )
    # elif extension == ".pptx":
    #     ppt_documents = extract_pptx(file_path)
    #     documents.extend(ppt_documents)
    elif extension == ".docx":
        text = extract_docx(file_path)
        documents.append(
            {
                "content":text,
                "type": "text"
            }
        )
    else:
        raise ValueError(f"Unsupported file extension {extension}")
    return documents


In [97]:
def chunk_text(text,chunk_size=1000,overlap=200):
    chunks = []
    start =0
    while start<len(text):
        end = start+chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks    

In [98]:
def load_new_folder(folder_path,collection):
    processed_files= get_processed_files(collection)
    all_chunks = []
    for file in os.listdir(folder_path):
        if file in processed_files:
            print(f"Skipping {file} (already embedded andstored)")
            continue
        file_path = os.path.join(
            folder_path,
            file
        )
        print(f"Reading file {file}")
        
        documents = load_document(file_path)
        for doc in documents:
            chunks = chunk_text(doc["content"])
            for i,chunk in enumerate(chunks):
                all_chunks.append(
                    {
                        "source":file,
                        "chunk_id": i,
                        "content": chunk,
                        "type":doc["type"]
                    }
                )
        print(f"Loaded{file}")
    return all_chunks

In [99]:
new_chunks = load_new_folder("../Data",collection)
print(
    f"Total Chunks: {len(new_chunks)}"
)

Reading file M1.pdf


LoadedM1.pdf
Reading file M3.pdf
LoadedM3.pdf
Skipping MLDOC.txt (already embedded andstored)
Reading file M4.pdf
LoadedM4.pdf
Skipping M5.pdf (already embedded andstored)
Reading file M2 (2).docx
LoadedM2 (2).docx
Total Chunks: 188


In [100]:
def create_embedding(text):
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents = text
    )
    return response.embeddings[0].values

In [101]:

for chunk in new_chunks:
    embedding = create_embedding(chunk["content"])
    collection.add(
        ids = [f"{chunk['source']}_{chunk['chunk_id']}"],
        documents = [chunk["content"]],
        embeddings = [embedding],
        metadatas=[
            {
                "source" : chunk["source"],
                "chunk_id" :chunk["chunk_id"],
                "type": chunk["type"]
            }
        ]
       )
print("stored successfully")


stored successfully


In [102]:
def retrieve_chunks(query):
    query_embedding = create_embedding(query)
    results = collection.query(
        query_embeddings =[query_embedding],
        n_results = 3
    )
    return{
        "documents":results["documents"][0],
        "metadata": results["metadatas"][0]
    }

In [103]:
results = retrieve_chunks(
    "What is ?"
)

print(results["metadata"][0])

{'chunk_id': 27, 'source': 'MLDOC.txt', 'type': 'text'}


In [104]:
def build_context(results):
    context = ""
    for doc in results["documents"][0]:
        context += doc
        context += "\n\n"
    return context

In [105]:
def detect_intent(question):
    question = question.lower()
    if "mcq" in question or "multiple choice questions" in question:
        return "mcq"
    elif "summary" in question or "summarize" in question:
        return "summary"
    elif "imp question" in question or "important question" in question:
        return "important_question"
    elif "solve" in question or "assignment" in question:
        return "assignment"
    else:
        return "explanation"

In [106]:
print(detect_intent("give a 10 MCQ's on supervised"))

mcq


In [107]:
def create_prompt(intent, context, question):

    if intent == "mcq":

        return f"""
            You are an AI Academic Assistant.

            Using ONLY the provided context,
            generate 10 MCQs.

            For each MCQ provide:

            1. Question
            2. Four Options
            3. Correct Answer
            4. Explanation

            Context:
            {context}

            Student Request:
            {question}
            """

    elif intent == "summary":

        return f"""
            You are an AI Academic Assistant.

            Create concise exam notes.

            Include:

            - Key Concepts
            - Important Definitions
            - Important Points

            Context:
            {context}

            Student Request:
            {question}
            """

    elif intent == "important_questions":

        return f"""
            You are an AI Academic Assistant.

            Generate likely university exam questions.

            Provide:

            - 2 Marks Questions
            - 5 Marks Questions
            - 10 Marks Questions

            Context:
            {context}

            Student Request:
            {question}
            """

    elif intent == "assignment":

        return f"""
            You are an AI Academic Assistant.

            Solve the assignment question step by step.

            Use only the provided context.

            Context:
            {context}

            Student Request:
            {question}
            """

    else:

        return f"""
            You are an AI Academic Assistant.

            Explain clearly in simple student-friendly language.

            Context:
            {context}

            Student Question:
            {question}
            """

In [116]:
def ask_question(query):
    
    results = retrieve_chunks(query)
    context = build_context(results)
    metadata = results["metadata"]
    intent = detect_intent(query)
    # Step 5: Create prompt
    prompt = create_prompt(
        intent,context,query
    ) 
     

    # Step 6: Generate answer
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    sources = []
    for item in metadata:
        sources.append(
            f"{item['source']} (Chunk {item['chunk_id']})"
        )
    return {
        "answer":response.text,
        "sources": sources
        }
    

In [117]:
while True:
    query = input("\nEnter a Question:")
    if query.lower()== "exit":
        break
    Answer = ask_question(query)
    display(Markdown(Answer["answer"]))
    print("\nSources:")

    for source in Answer["sources"]:
        print("-", source)

Alright, let's break down the DBSCAN algorithm in a way that's easy to understand!

### What is the DBSCAN Algorithm?

Imagine you have a big map with lots of dots (data points) on it, and you want to find groups of dots that are clustered together. DBSCAN, which stands for **D**ensity-**B**ased **S**patial **C**lustering of **A**pplications with **N**oise, is a smart way to do this.

Unlike some other grouping methods that try to fit dots into a certain number of predefined groups, DBSCAN looks for "dense" regions in your data. It basically says: "If there are enough dots really close to each other, that's a cluster!" It's also great because it can spot dots that don't belong to any group – we call these "noise" or "outliers."

To do its job, DBSCAN needs two main "settings" or parameters:

*   **`eps` (epsilon):** Think of this as a **radius** or a "search distance." For any given dot, `eps` tells DBSCAN how far to look around that dot to see if there are other dots nearby. It defines what "nearby" means.
*   **`MinPts` (Minimum Points):** This is the **minimum number of dots** that need to be found within the `eps` radius of a dot for that area to be considered "dense." If you don't find at least `MinPts` dots (including the dot itself), that area isn't dense enough to start a cluster.

There are some helpful hints for setting `MinPts`:
*   A common rule is to set `MinPts` to be at least `D + 1`, where `D` is the number of "features" or "dimensions" your data has (e.g., if you have 2D points on a map, `D=2`).
*   Often, a value of `MinPts = 3` is a good starting point for many datasets.

---

### Analyzing the Steps in the DBSCAN Algorithm

DBSCAN works by classifying each data point into one of three types, and then using these types to form clusters. Let's look at the steps:

#### **Step 1: Identify Core Points**

This is where DBSCAN finds the "hearts" or "centers" of potential clusters.

*   **How it works:** For every single dot in your dataset, DBSCAN draws a circle (with radius `eps`) around it. Then, it counts how many other dots fall inside that circle.
*   **What makes it a Core Point:** If the number of dots inside the circle (including the original dot) is equal to or greater than your `MinPts` setting, then that original dot is declared a **Core Point**.
*   **Analogy:** Imagine a busy city center. If you stand in the middle (`eps` radius) and see many people (`MinPts`) around you, you're in a dense, "core" part of the city. These core points are the building blocks for our clusters.

#### **Step 2: Form Clusters**

Once we've found all the core points, DBSCAN starts building the actual clusters.

*   **How it works:**
    1.  DBSCAN picks a core point that hasn't been assigned to a cluster yet.
    2.  It then starts a brand new cluster with this core point.
    3.  From this core point, it reaches out and "collects" all the other points that are "density-connected" to it. These collected points are added to the new cluster.
    4.  It keeps doing this, expanding the cluster outwards like a ripple, until no more density-connected points can be found.
*   **Analogy:** A core point is like a starting point for a neighborhood. It pulls in all its immediate neighbors (within `eps`), and if those neighbors are also core points, they pull in *their* neighbors, and so on, building up the entire neighborhood.

#### **Step 3: Density Connectivity**

This is a crucial concept for how clusters expand.

*   **What it means:** Two dots, let's call them 'A' and 'B', are "density-connected" if you can draw a path between them through a series of other dots, where each dot in the path is within `eps` of the next. **Crucially, at least one dot in this path (usually the starting core point) must be a core point.**
*   **Why it's important:** This allows clusters to form irregular shapes and connect dots that aren't directly next to each other, as long as there's a "bridge" of dense points between them. For example, a dot might not be a core point itself (it might not have enough neighbors), but if it's within the `eps` radius of a core point, it will be included in that core point's cluster. These points are sometimes called "border points."
*   **Analogy:** Imagine a string of beads. If each bead is close enough to the next (within `eps`), and at least one bead is a "core" bead (meaning it has many neighbors itself), then all beads in that string belong to the same cluster.

#### **Step 4: Label Noise Points**

After all core points have been processed and all possible clusters have been formed, there might be some dots left over.

*   **What they are:** Any dot that wasn't identified as a core point, and also wasn't "density-connected" (within `eps`) to any core point, is considered a **Noise Point**.
*   **Importance:** This is a big advantage of DBSCAN! It automatically identifies outliers or "noise" in your data that don't fit into any of the dense clusters.
*   **Analogy:** These are the isolated houses or lonely travelers that aren't part of any town or village on your map. They stand alone.

---

In summary, DBSCAN is a powerful and flexible clustering algorithm that builds groups based on how densely packed data points are. It's excellent for finding clusters of unusual shapes and for identifying data points that are simply outliers.


Sources:
- M5.pdf (Chunk 6)
- M5.pdf (Chunk 7)
- M5.pdf (Chunk 5)
